# Dataset Exploration and Preprocessing

This notebook focuses on exploring and preparing the student face dataset for deep learning model development. The objective is to understand dataset characteristics, inspect class distributions, visualize samples, and apply preprocessing techniques required for robust model training.

Key tasks performed include:

* Dataset inspection and visualization 
* Student class analysis
* Image normalization
* RGB conversion
* Image resizing
* Dataset balancing
* Preparation of data for CNN training

The resulting processed dataset serves as the foundation for student face recognition and future AI-powered attendance applications.


In [ ]:
import os
import random
from collections import Counter
from shutil import copy2

import torch
import torchvision
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import PIL

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.notebook import tqdm

In [ ]:
print("torch version : ", torch.__version__)
print("torchvision version : ", torchvision.__version__)
print("numpy version : ", np.__version__)
print("matplotlib version : ", matplotlib.__version__)

!python --version

torch version :  2.11.0+cpu
torchvision version :  0.26.0+cpu
numpy version :  2.1.3
matplotlib version :  3.10.0
Python 3.13.5


In [ ]:
data_dir = "data"
train_dir = os.path.join(data_dir)

print("Data directory: ", train_dir)

Data directory:  data


In [ ]:
classes = os.listdir(train_dir)
classes

['abdulrahman', 'fatima', 'labaran', 'maryam', 'usman', 'zainab']

In [ ]:
def sample_images(data_path, classname):
    # Gets the files in the directory
    class_dir = os.path.join(data_path, classname)
    if not os.path.exists(class_dir):
        return "Invalid directory"
    image_list = os.listdir(class_dir)
    if len(image_list) < 4:
        return "Not enough images in folder"

    # Pick four random images
    images_sample = random.sample(image_list, 4)

    # Plot them
    plt.figure(figsize=(20, 20))
    for i in range(4):
        img_loc = os.path.join(class_dir, images_sample[i])
        img = PIL.Image.open(img_loc)
        plt.subplot(1, 4, i + 1)
        plt.imshow(img)
        plt.axis("off")

In [ ]:
sample_images(data_dir, "usman")

In [ ]:
sample_images(data_dir, "abdulrahman")

In [ ]:
sample_images(data_dir, "maryam")

In [ ]:
sample_images(data_dir, "zainab")

In [ ]:
sample_images(data_dir, "labaran")

In [ ]:
sample_images(data_dir, "fatima")

# Preparing Our Data

In [6]:
class ConvertToRGB(object):
    def __call__(self, img):
        if img.mode != "RGB":
            img.convert("RGB")
        return img

In [7]:
transform_basic = transforms.Compose(
    [
        ConvertToRGB(),
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ]
)

In [8]:
batch_size = 32
dataset = datasets.ImageFolder(root=train_dir, transform=transform_basic)
dataset_loader = DataLoader(dataset, batch_size=batch_size)

batch_shape = next(iter(dataset_loader))[0].shape
print("Getting batches of shape: ", batch_shape)

Getting batches of shape:  torch.Size([32, 3, 224, 224])


# Normalized Data

Normalized data made our network perform better. That is data with a mean of $0$ and a standard deviation of $1$. We'll want to normalize this data.

In [9]:
def get_mean_std(loader):
    """Computes the mean and standard deviation of image data.

    Input: a `DataLoader` producing tensors of shape [batch_size, channels, pixels_x, pixels_y]
    Output: the mean of each channel as a tensor, the standard deviation of each channel as a tensor
            formatted as a tuple (means[channels], std[channels])"""

    channels_sum, channels_squared_sum, num_batches = 0, 0, 0
    for data, _ in tqdm(loader, desc="Computing mean and std", leave=False):
        channels_sum += torch.mean(data, dim=[0, 2, 3])
        channels_squared_sum += torch.mean(data**2, dim=[0, 2, 3])
        num_batches += 1
    mean = channels_sum / num_batches
    std = (channels_squared_sum / num_batches - mean**2) ** 0.5

    return mean, std

In [10]:
mean, std = get_mean_std(dataset_loader)

print(f"Mean: {mean}")
print(f"Standard deviation: {std}")

Computing mean and std:   0%|          | 0/44 [00:00<?, ?it/s]

Mean: tensor([0.6665, 0.6047, 0.5631])
Standard deviation: tensor([0.2257, 0.2515, 0.2687])


In [11]:
transform_norm = transforms.Compose(
    [
        ConvertToRGB(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ]
)

In [12]:
norm_dataset = datasets.ImageFolder(root=train_dir, transform=transform_norm)
norm_loader = DataLoader(norm_dataset, batch_size)

batch_shape = next(iter(norm_loader))[0].shape
print("Getting batches of shape: ", batch_shape)

Getting batches of shape:  torch.Size([32, 3, 224, 224])


In [13]:
norm_mean, norm_std = get_mean_std(norm_loader)

print(f"Mean: {norm_mean}")
print(f"Standard deviation: {norm_std}")

Computing mean and std:   0%|          | 0/44 [00:00<?, ?it/s]

Mean: tensor([-2.6704e-07, -1.6015e-07,  1.6273e-07])
Standard deviation: tensor([1.0000, 1.0000, 1.0000])


# Train-Validation Split

In [14]:
train_dataset, val_dataset = random_split(norm_dataset, [0.8, 0.2])

length_train = len(train_dataset)
length_val = len(val_dataset)
length_dataset = len(norm_dataset)
percent_train = np.round(100 * length_train / length_dataset, 2)
percent_val = np.round(100 * length_val / length_dataset, 2)

print(f"Train data is {percent_train}% of full data")
print(f"Validation data is {percent_val}% of full data")

Train data is 80.04% of full data
Validation data is 19.96% of full data


In [5]:
# from training import class_counts

def class_counts(dataset):
    c = Counter(x[1] for x in tqdm(dataset))
    class_to_index = dataset.dataset.class_to_idx
    return pd.Series({cat: c[idx] for cat, idx in class_to_index.items()})


In [16]:
train_count = class_counts(train_dataset)
train_count

  0%|          | 0/1115 [00:00<?, ?it/s]

abdulrahman    213
fatima         162
labaran        196
maryam         171
usman          159
zainab         214
dtype: int64

In [ ]:
train_count.plot(kind="bar");

In [2]:
val_count = class_counts(val_dataset)

NameError: name 'val_dataset' is not defined

In [ ]:
val_counts.plot(kind="bar");

# Unbalance Classes

In [18]:
def undersample_dataset(dataset_dir, output_dir, target_count=None):
    """
    Undersample the dataset to have a uniform distribution across classes.

    Parameters:
    - dataset_dir: Path to the directory containing the class folders.
    - output_dir: Path to the directory where the undersampled dataset will be stored.
    - target_count: Number of instances to keep in each class. If None, the class with the least instances will set the target.
    """
    # Mapping each class to its files
    classes_files = {}
    for class_name in os.listdir(dataset_dir):
        class_dir = os.path.join(dataset_dir, class_name)
        if os.path.isdir(class_dir):
            files = os.listdir(class_dir)
            classes_files[class_name] = files

    # Determine the minimum class size if target_count is not set
    if target_count is None:
        target_count = min(len(files) for files in classes_files.values())

    # Creating the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Perform undersampling
    for class_name, files in classes_files.items():
        print("Copying images for class", class_name)
        class_output_dir = os.path.join(output_dir, class_name)
        if not os.path.exists(class_output_dir):
            os.makedirs(class_output_dir)

        # Randomly select target_count images
        selected_files = random.sample(files, min(len(files), target_count))

        # Copy selected files to the output directory
        for file_name in tqdm(selected_files):
            src_path = os.path.join(dataset_dir, class_name, file_name)
            dst_path = os.path.join(class_output_dir, file_name)
            copy2(src_path, dst_path)

    print(f"Undersampling completed. Each class has up to {target_count} instances.")

In [6]:
output_dir = os.path.join("data_p1", "data_undersampled", "train")
print("Output directory:", output_dir)

Output directory: data_p1\data_undersampled\train


In [20]:
undersample_dataset(train_dir, output_dir)

Copying images for class abdulrahman


  0%|          | 0/200 [00:00<?, ?it/s]

Copying images for class fatima


  0%|          | 0/200 [00:00<?, ?it/s]

Copying images for class labaran


...

Copying images for class usman


  0%|          | 0/200 [00:00<?, ?it/s]

Copying images for class zainab


  0%|          | 0/200 [00:00<?, ?it/s]

Undersampling completed. Each class has up to 200 instances.


In [7]:
undersampled_dataset = datasets.ImageFolder(root=output_dir)
undersampled_dataset.classes

['abdulrahman', 'fatima', 'labaran', 'maryam', 'usman', 'zainab']

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

under_counts = class_counts(undersample_dataset)

NameError: name 'undersample_dataset' is not defined

In [1]:
under_counts.plot(kind="bar", ax=ax)

NameError: name 'under_counts' is not defined